In [ ]:
from pathlib import Path
import json
from PIL import Image
from tqdm.auto import tqdm

with open("../AGAR_dataset/training_lists/higher_resolution_train.txt") as f:
    train_ids = json.load(f)
with open("../AGAR_dataset/training_lists/higher_resolution_val.txt") as f:
    val_ids = json.load(f)

splits = ["train", "val"]

split_data = {
    "train": [id for id in train_ids if id.isdigit()],
    "val": [id for id in val_ids if id.isdigit()],
}


for split in splits:
    folder = Path("../datasets/agar_coco")
    images_folder = folder / "data"

    folder.mkdir(exist_ok=True, parents=True)
    images_folder.mkdir(exist_ok=True)

    labels = {
        "images": [],
        "categories": [{"id": 1, "name": "colony"}],
        "annotations": [],
    }
    annotation_id = 0

    for id, image_id in tqdm(
        enumerate(split_data[split]),
        desc=split,
        total=len(split_data[split]),
    ):
        image_filename = f"{image_id}.jpg"

        image = Image.open(f"../AGAR_dataset/dataset/{image_filename}")
        image.save(images_folder / image_filename)

        height = image.height
        width = image.width

        labels["images"].append(
            {
                "id": id,
                "file_name": image_filename,
                "height": height,
                "width": width,
            }
        )

        with open(f"../AGAR_dataset/dataset/{image_id}.json") as f:
            image_label_data = json.load(f)

        for label in image_label_data["labels"]:
            labels["annotations"].append(
                {
                    "id": annotation_id,
                    "image_id": id,
                    "category_id": 1,
                    "area": label["width"] * label["height"],
                    "bbox": [label["x"], label["y"], label["width"], label["height"]],
                    "iscrowd": 0,
                }
            )

            annotation_id += 1

    with open(folder / f"{split}.json", "w") as f:
        json.dump(labels, f)

In [ ]:
%pip install fiftyone

In [ ]:
import fiftyone as fo

dataset_train = fo.Dataset.from_dir(
    "../datasets/agar_coco",
    labels_path="train.json",
    dataset_type=fo.types.COCODetectionDataset,
)

session = fo.launch_app(dataset_train, auto=False, address="0.0.0.0", port=5151)

print(f"Dataset launched at http://{session.server_address}:{session.server_port}")